# 05 — Final Data Load & KPI Preparation

**Objective:** Compute KPIs and prepare aggregated datasets for the Tableau dashboard.

**Input:** `data/processed/cleaned_online_sales.csv`

**Outputs:**
- `data/processed/kpi_summary.csv`
- `data/processed/monthly_trends.csv`
- `data/processed/category_summary.csv`
- `data/processed/country_summary.csv`
- `data/processed/product_summary.csv`

In [ ]:
import pandas as pd
import numpy as np
import os

OUT_DIR = '../data/processed/'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv('../data/processed/cleaned_online_sales.csv', parse_dates=['InvoiceDate'])
print(f'Loaded {df.shape[0]:,} rows')

## 1. Overall KPIs

In [ ]:
total_orders = len(df)
total_returns = df['IsReturn'].sum()
return_rate = df['IsReturn'].mean() * 100
total_revenue = df['Revenue'].sum()
returned_revenue = df[df['IsReturn'] == 1]['Revenue'].sum()
net_revenue = total_revenue - returned_revenue
avg_order_value = df['Revenue'].mean()
unique_customers = df['CustomerID'].nunique()
unique_products = df['StockCode'].nunique()

kpis = {
    'Total Orders': [total_orders],
    'Total Returns': [total_returns],
    'Return Rate (%)': [round(return_rate, 2)],
    'Total Revenue': [round(total_revenue, 2)],
    'Returned Revenue': [round(returned_revenue, 2)],
    'Net Revenue': [round(net_revenue, 2)],
    'Avg Order Value': [round(avg_order_value, 2)],
    'Unique Customers': [unique_customers],
    'Unique Products': [unique_products]
}

kpi_df = pd.DataFrame(kpis)
display(kpi_df.T.rename(columns={0: 'Value'}))

kpi_df.to_csv(OUT_DIR + 'kpi_summary.csv', index=False)
print('Saved: kpi_summary.csv')

## 2. Monthly Trends (for Tableau time-series)

In [ ]:
monthly = df.groupby(df['InvoiceDate'].dt.to_period('M')).agg(
    total_orders=('IsReturn', 'count'),
    returns=('IsReturn', 'sum'),
    return_rate=('IsReturn', 'mean'),
    total_revenue=('Revenue', 'sum'),
    returned_revenue=('Revenue', lambda x: x[df.loc[x.index, 'IsReturn'] == 1].sum()),
    avg_order_value=('Revenue', 'mean'),
    unique_customers=('CustomerID', 'nunique')
).reset_index()
monthly['InvoiceDate'] = monthly['InvoiceDate'].astype(str)
monthly['return_rate'] = (monthly['return_rate'] * 100).round(2)
monthly['net_revenue'] = monthly['total_revenue'] - monthly['returned_revenue']

display(monthly.head())
monthly.to_csv(OUT_DIR + 'monthly_trends.csv', index=False)
print('Saved: monthly_trends.csv')

## 3. Category Summary

In [ ]:
cat_summary = df.groupby('Category').agg(
    total_orders=('IsReturn', 'count'),
    returns=('IsReturn', 'sum'),
    return_rate=('IsReturn', 'mean'),
    total_revenue=('Revenue', 'sum'),
    avg_price=('UnitPrice', 'mean'),
    avg_discount=('Discount', 'mean')
).reset_index()
cat_summary['return_rate'] = (cat_summary['return_rate'] * 100).round(2)
cat_summary['revenue_at_risk'] = df[df['IsReturn'] == 1].groupby('Category')['Revenue'].sum().values

display(cat_summary)
cat_summary.to_csv(OUT_DIR + 'category_summary.csv', index=False)
print('Saved: category_summary.csv')

## 4. Country Summary

In [ ]:
country_summary = df.groupby('Country').agg(
    total_orders=('IsReturn', 'count'),
    returns=('IsReturn', 'sum'),
    return_rate=('IsReturn', 'mean'),
    total_revenue=('Revenue', 'sum'),
    unique_customers=('CustomerID', 'nunique')
).reset_index()
country_summary['return_rate'] = (country_summary['return_rate'] * 100).round(2)

display(country_summary)
country_summary.to_csv(OUT_DIR + 'country_summary.csv', index=False)
print('Saved: country_summary.csv')

## 5. Product Summary

In [ ]:
prod_summary = df.groupby(['StockCode', 'Description', 'Category']).agg(
    total_orders=('IsReturn', 'count'),
    returns=('IsReturn', 'sum'),
    return_rate=('IsReturn', 'mean'),
    total_revenue=('Revenue', 'sum'),
    avg_price=('UnitPrice', 'mean')
).reset_index()
prod_summary['return_rate'] = (prod_summary['return_rate'] * 100).round(2)
prod_summary = prod_summary.sort_values('returns', ascending=False)

display(prod_summary.head(10))
prod_summary.to_csv(OUT_DIR + 'product_summary.csv', index=False)
print('Saved: product_summary.csv')

## 6. Files Ready for Tableau

| File | Purpose |
|------|--------|
| `cleaned_online_sales.csv` | Full cleaned dataset |
| `kpi_summary.csv` | KPI cards |
| `monthly_trends.csv` | Time-series charts |
| `category_summary.csv` | Category breakdown |
| `country_summary.csv` | Geographic analysis |
| `product_summary.csv` | Product-level detail |

All files saved to `data/processed/`. Import into Tableau Public.